# Exercise 2: Allomorphy tables

*Morphological Structure · version 0.1.0*

> For the Turkish dative suffix (*-a/-e*) and the locative suffix (*-da/-de/-ta/-te*), fill in the correct allomorph for each of the following roots: *ev* (house), *araba* (car), *göz* (eye), *kapı* (door), *çiçek* (flower), *kuş* (bird). State the rule governing each allomorph choice.

**Setup.** `pip install "turkicnlp[hfst]==0.5.0"`. Where the compiled `hfst` package has no wheel for your Python version, install `turkicnlp==0.5.0 pyhfst` instead: the first code cell then uses the pure-Python `pyhfst` reader, which has the same lookup interface. The Apertium transducers are downloaded once with `turkicnlp.download`. With `pyhfst`, readings may be listed in a different order than with `hfst`, and the pipeline may select a different reading when several have the same weight.

The code applies the chapter's rules: the last vowel of the stem chooses *a* or *e* (two-way harmony), a voiceless final consonant chooses *t* over *d* in the locative, a vowel-final stem takes the buffer *y* before the dative, and *çiçek* is marked in the lexicon as a stem whose final *k* becomes *ğ* before a vowel. Each generated form is then looked up in the Apertium Turkish transducer.

In [ ]:
import sys
try:
    import hfst                                   # compiled HFST, as in the chapter
except ImportError:                               # fall back to the pure-Python reader
    import pyhfst
    sys.modules["hfst"] = pyhfst

import turkicnlp
from turkicnlp import Pipeline

def pipeline(lang):
    turkicnlp.download(lang, processors=["morph"])          # Apertium transducers (once)
    return Pipeline(lang, processors=["tokenize", "morph"], tokenize_backend="rule", morph_backend="apertium")

def analyser(pipe):
    """The Apertium processor of a pipeline (private API in 0.5.0)."""
    return next(p for p in pipe._processors if hasattr(p, "_analyze"))

def readings(pipe, word, plus=False):
    """All transducer readings of one word; by default without '+' continuations (copula etc.)."""
    out = [r["raw"] for r in analyser(pipe)._analyze(word)]
    return out if plus else [r for r in out if "+" not in r]

def show(pipe, sentence):
    for w in pipe(sentence).words:
        print(f"  {w.text:<18} lemma={w.lemma or '_':<10} upos={w.upos or '_':<6} feats={w.feats or '_'}")

tur = pipeline("tur")

In [ ]:
BACK, FRONT = set("aıou"), set("eiöü")
VOICELESS = set("çfhkpsşt")
SOFTENING = {"çiçek": "çiçeğ"}        # lexically marked stems whose final k becomes ğ before a vowel

def last_vowel(stem):
    return next(c for c in reversed(stem) if c in BACK | FRONT)

def dative(stem):
    a = "a" if last_vowel(stem) in BACK else "e"
    if stem[-1] in BACK | FRONT:
        return stem + "y" + a, "vowel-final stem: buffer y; " + ("back" if a == "a" else "front") + " last vowel"
    return SOFTENING.get(stem, stem) + a, ("k > ğ before the vowel; " if stem in SOFTENING else "") + \
        ("back" if a == "a" else "front") + " last vowel"

def locative(stem):
    a = "a" if last_vowel(stem) in BACK else "e"
    d = "t" if stem[-1] in VOICELESS else "d"
    return stem + d + a, ("voiceless" if d == "t" else "voiced or vowel") + " final sound; " + \
        ("back" if a == "a" else "front") + " last vowel"

def accepted(word, tag):
    return any(f"<{tag}>" in r for r in readings(tur, word))

for stem in ["ev", "araba", "göz", "kapı", "çiçek", "kuş"]:
    (dat, why_dat), (loc, why_loc) = dative(stem), locative(stem)
    print(f"{stem:7} DAT {dat:8} ({why_dat}; Apertium: {accepted(dat, 'dat')})")
    print(f"{'':7} LOC {loc:8} ({why_loc}; Apertium: {accepted(loc, 'loc')})")
print("Unsoftened *çiçeke*:", readings(tur, "çiçeke"), "| çiçeği:", readings(tur, "çiçeği"))

## Worked solution

Output of `experiments/ch05_exercises/run_ch05_exercises.py` on 19 October 2026 (TurkicNLP 0.5.0, HFST 3.16.0.1, Python 3.11):

```
ev      DAT eve      (front last vowel; Apertium: True)
        LOC evde     (voiced or vowel final sound; front last vowel; Apertium: True)
araba   DAT arabaya  (vowel-final stem: buffer y; back last vowel; Apertium: True)
        LOC arabada  (voiced or vowel final sound; back last vowel; Apertium: True)
göz     DAT göze     (front last vowel; Apertium: True)
        LOC gözde    (voiced or vowel final sound; front last vowel; Apertium: True)
kapı    DAT kapıya   (vowel-final stem: buffer y; back last vowel; Apertium: True)
        LOC kapıda   (voiced or vowel final sound; back last vowel; Apertium: True)
çiçek   DAT çiçeğe   (k > ğ before the vowel; front last vowel; Apertium: False)
        LOC çiçekte  (voiceless final sound; front last vowel; Apertium: True)
kuş     DAT kuşa     (back last vowel; Apertium: True)
        LOC kuşta    (voiceless final sound; back last vowel; Apertium: True)
Unsoftened *çiçeke*: ['çiçek<n><dat>'] | çiçeği: []
```

| Root | Dative | Locative | Rule |
|---|---|---|---|
| *ev* | *eve* | *evde* | front last vowel → *e*; voiced *v* → *d* |
| *araba* | *arabaya* | *arabada* | vowel-final → buffer *y* in the dative; back → *a*; vowel → *d* |
| *göz* | *göze* | *gözde* | front → *e*; voiced *z* → *d* |
| *kapı* | *kapıya* | *kapıda* | vowel-final → *y*; back → *a*; vowel → *d* |
| *çiçek* | *çiçeğe* | *çiçekte* | front → *e*; final *k* becomes *ğ* before a vowel (a lexical property); voiceless *k* → *t* |
| *kuş* | *kuşa* | *kuşta* | back → *a*; voiceless *ş* → *t* |

The transducer accepts eleven of the twelve forms. It rejects *çiçeğe* and accepts the incorrect *çiçeke* as the dative instead; it also rejects *çiçeği*. In this version of the Apertium Turkish lexicon, *çiçek* is therefore not marked for the *k*/*ğ* alternation that *kitap*/*kitab-* and *çocuk*/*çocuğ-* have. A transducer check confirms forms only as far as its lexicon is right: here the rule is right and the transducer is wrong. This is the stem-final alternation that the chapter says "must be tracked in the morphological lexicon as a property of the stem".

## Your turn — extension

1. Add the ablative (*-dan/-den/-tan/-ten*) and check it the same way.
2. Add *kitap* and *ağaç*. Which lexical marking do they need for the dative?
3. Add *saat* 'hour' and *rol* 'role', two exceptions from the vowel harmony section of The Computational Phenotype. What does the rule produce, and what does the transducer accept?